# Week 3 exercises: Loading Data, Databases, APIs, and Joining & Reshaping

*Complete all exercises in this notebook during week 3. The exercises are not submitted — this is purely for your own learning and for preparing for the assignments, project and exam.*

Note! You should always complete the tasks first without using AI. If you need AI to get through these tasks, then you will not be able to pass the oral exam. AI can be used to check if your code could be simplified or improved after you solve the task.

---

## Part 1: Storing API Keys Securely

### Exercise 1: Creating and using a .env file

When working with APIs that require authentication, you need an API key. API keys must **never** be written directly in your code or notebook, because:
- If you upload your notebook to GitHub, anyone can see and [misuse your key](https://www.theregister.com/2026/03/03/gemini_api_key_82314_dollar_charge/).
- Sharing your notebook with a classmate would also expose the key (what if they accidentally share it on Github?).

A simple and safe approach is to store your key in a separate `.env` (env for environment) file that is **excluded from version control**.

a) Create a file called `.env` in the same folder as this notebook (you can do this with a text editor, VS Code, or from Python). I recommend [Sublime Text](https://www.sublimetext.com/) as a convenient, light and free way to view and create practically any file needed in software development. The file should have the following content:

```
# .env (local only, never commit)
GEMINI_API_KEY=YOUR_API_KEY_HERE
```

Replace `YOUR_API_KEY_HERE` with your actual API key. For this exercise, we will use Google's Gemini API as an example (you can obtain one from [Google AI Studio](https://aistudio.google.com/apikey)).

b) Add `.env` to your `.gitignore` file so that the key is never uploaded to GitHub. You can do this in GitHub Desktop by right clicking the file and selecting Ignore File, which will add the file to your .gitignore file. Alternatively, you can manually open (or create) `.gitignore` in a text editor and adding a line containing `.env`. **Do not skip this step.**

## Exercise 2: Reading the API key from the .env file

The `python-dotenv` package automates reading variables that store information such as API keys from the .env file and is the standard approach in Python projects. It loads all variables from a `.env` file into the environment in a single call.

Run the code below to make sure you have stored the API key correctly.

**Note:** You may need to install the package first with `pip install python-dotenv`. 

In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

# Load all variables from .env into the environment
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")

if api_key:
    print(f"Key loaded: {api_key[:3]}...")
else:
    print("ERROR: GEMINI_API_KEY not found. Check your .env file.")

Key loaded: AQ....


## Part 2: Loading and Exporting Data

### Exercise 3: Loading CSV data

The file `data/cafe_sales.csv` contains sales data for a small café.

a) Load the file into a pandas DataFrame. Print the shape and the first 5 rows.

b) Load only the columns `product`, `category`, and `revenue` using the `usecols` parameter. Set `product` as the index. Print the resulting DataFrame.

c) Export the DataFrame from (b) to a new CSV file called `data/category_revenue.csv` without the index column. Then reload the exported file and print it to verify the export was correct.

In [3]:
import numpy as np
import pandas as pd

#a)
df = pd.read_csv("../data/cafe_sales.csv")
print("Shape:", df.shape)
print()
df.head()

Shape: (12, 5)



,product,category,units_sold,unit_price,revenue
0,Espresso,Coffee,320,3.5,1120.0
1,Cappuccino,Coffee,280,4.5,1260.0
2,Latte,Coffee,250,4.9,1225.0
3,Green Tea,Tea,180,3.2,576.0
4,Chai Latte,Tea,150,4.7,705.0


In [4]:
#b)
df_subset = pd.read_csv("../data/cafe_sales.csv", usecols = ['product', 'category', 'revenue'], index_col = ['product'])
print(df)

             product category  units_sold  unit_price  revenue
0           Espresso   Coffee         320         3.5   1120.0
1         Cappuccino   Coffee         280         4.5   1260.0
2              Latte   Coffee         250         4.9   1225.0
3          Green Tea      Tea         180         3.2    576.0
4         Chai Latte      Tea         150         4.7    705.0
5          Croissant   Pastry         200         3.8    760.0
6   Blueberry Muffin   Pastry         170         4.2    714.0
7      Cinnamon Roll   Pastry         130         4.5    585.0
8     Iced Americano   Coffee         210         4.0    840.0
9       Matcha Latte      Tea         120         5.2    624.0
10             Bagel   Pastry         160         3.5    560.0
11        Flat White   Coffee         190         4.6    874.0


In [5]:
#c)
df_subset.to_csv("../data/category_revenue.csv", index=False)

df_check = pd.read_csv("../data/category_revenue.csv")
print(df_check)


   category  revenue
0    Coffee   1120.0
1    Coffee   1260.0
2    Coffee   1225.0
3       Tea    576.0
4       Tea    705.0
5    Pastry    760.0
6    Pastry    714.0
7    Pastry    585.0
8    Coffee    840.0
9       Tea    624.0
10   Pastry    560.0
11   Coffee    874.0


### Exercise 4: Working with JSON data

The file `data/projects.json` contains project records for a small company. Each project has a list of tasks with estimated hours and a status.

a) Load the file using the `json` module. Print the number of projects and the full first record.

b) Use `pd.json_normalize()` to flatten the nested tasks into a regular DataFrame. The result should have one row per task and include the `project_id` and `title` from the parent record. Print the DataFrame. The documentation for pd.json_normalice() is found [here](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).

c) Export the flattened tasks DataFrame to a JSON file called `data/tasks_flat.json` using the `records` orientation. Load it again and print it to see that it was saved correctly.

In [6]:
#a)
import json
with open("../data/projects.json") as f:
    projects = json.load(f)
    print(f"The number of projects is {len(projects)}")
    print()
    print("The  first record:")
    print(projects[0])

The number of projects is 3

The  first record:
{'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura', 'tasks': [{'task': 'Wireframing', 'hours': 20, 'status': 'completed'}, {'task': 'Frontend development', 'hours': 60, 'status': 'in_progress'}, {'task': 'User testing', 'hours': 15, 'status': 'not_started'}]}


In [7]:
#b)
df_tasks = pd.json_normalize(
    projects, 
    record_path = "tasks", 
    meta = ["project_id", "title"])
print(df_tasks)

                   task  hours       status project_id  \
0           Wireframing     20    completed       P001   
1  Frontend development     60  in_progress       P001   
2          User testing     15  not_started       P001   
3             UI design     35    completed       P002   
4           Backend API     80  in_progress       P002   
5  App store submission      5  not_started       P002   
6          Beta testing     25  not_started       P002   
7        Schema mapping     15    completed       P003   
8           ETL scripts     45    completed       P003   
9            Validation     20  in_progress       P003   

                     title  
0         Website Redesign  
1         Website Redesign  
2         Website Redesign  
3        Mobile App Launch  
4        Mobile App Launch  
5        Mobile App Launch  
6        Mobile App Launch  
7  Data Pipeline Migration  
8  Data Pipeline Migration  
9  Data Pipeline Migration  


In [8]:
#c) using json
df_tasks.to_json("../data/tasks_flat.json", orient = "records", indent = 2)
with open ("../data/tasks_flat.json") as f:
    tasks_flat = json.load(f)
display(tasks_flat)
    

[{'task': 'Wireframing',
  'hours': 20,
  'status': 'completed',
  'project_id': 'P001',
  'title': 'Website Redesign'},
 {'task': 'Frontend development',
  'hours': 60,
  'status': 'in_progress',
  'project_id': 'P001',
  'title': 'Website Redesign'},
 {'task': 'User testing',
  'hours': 15,
  'status': 'not_started',
  'project_id': 'P001',
  'title': 'Website Redesign'},
 {'task': 'UI design',
  'hours': 35,
  'status': 'completed',
  'project_id': 'P002',
  'title': 'Mobile App Launch'},
 {'task': 'Backend API',
  'hours': 80,
  'status': 'in_progress',
  'project_id': 'P002',
  'title': 'Mobile App Launch'},
 {'task': 'App store submission',
  'hours': 5,
  'status': 'not_started',
  'project_id': 'P002',
  'title': 'Mobile App Launch'},
 {'task': 'Beta testing',
  'hours': 25,
  'status': 'not_started',
  'project_id': 'P002',
  'title': 'Mobile App Launch'},
 {'task': 'Schema mapping',
  'hours': 15,
  'status': 'completed',
  'project_id': 'P003',
  'title': 'Data Pipeline Migr

In [9]:
#c using pandas
df_check = pd.read_json("../data/tasks_flat.json")
display(df_check)

,task,hours,status,project_id,title
0,Wireframing,20,completed,P001,Website Redesign
1,Frontend development,60,in_progress,P001,Website Redesign
2,User testing,15,not_started,P001,Website Redesign
3,UI design,35,completed,P002,Mobile App Launch
4,Backend API,80,in_progress,P002,Mobile App Launch
5,App store submission,5,not_started,P002,Mobile App Launch
6,Beta testing,25,not_started,P002,Mobile App Launch
7,Schema mapping,15,completed,P003,Data Pipeline Migration
8,ETL scripts,45,completed,P003,Data Pipeline Migration
9,Validation,20,in_progress,P003,Data Pipeline Migration


## Part 3: Databases

### Exercise 5: Connecting and querying a SQLite database

The file `data/retail.db` is a SQLite database with tables including Product, Vendor, Store, Region, Customer, SalesTransaction, and Includes. The tables, relationships and a sample of the data are visualized below:

<img src="data/schema.png" alt="Database schema" width="700">


a) Connect to the database using `sqlite3`. List all table names by querying `sqlite_master`.

b) Using `fetchall()`, retrieve and print the names and prices of all products that cost more than 100€. Format the output so that each product is printed as `ProductName: Price€`.

c) Using a parameterised query (with `?` placeholders), retrieve all products supplied by vendor `"MK"`. Print the product names.

In [10]:
#a)
import sqlite3
connection = sqlite3.connect("../data/retail.db")
db = connection.cursor()

db.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = db.fetchall()

for table in tables:
    print(table[0])


vendor
category
product
region
store
customer
salestransaction
includes


In [11]:
#b)
product_expensive = db.execute(
    """
    SELECT ProductName, ProductPrice 
    FROM Product 
    WHERE ProductPrice > 100
    """
).fetchall()

for product in product_expensive:
    print(f" {product[0]}: {product[1]}€")

 Tiny Tent: 150€
 Biggy Tent: 250€
 Hi-Tec GPS: 300€
 Comfy Harness: 150€
 Sunny Charger: 125€
 Mega Camera: 275€
 Luxo Tent: 500€


In [12]:
#c)
vendor = "MK"
MK_products = db.execute(
    "SELECT ProductName FROM Product WHERE VendorID = ?", [vendor]
).fetchall()

for product in MK_products:
    print(product[0])

Easy Boot
Cosy Sock
Tiny Tent
Biggy Tent
Power Pedals
Comfy Harness
Strongster Carribeaner
Electra Compass


### Exercise 6: Loading database results into pandas

Using the same `retail.db` database:

a) Use `pd.read_sql_query()` to load the entire `Customer` table into a DataFrame. Print the shape and the first 5 rows.

b) Write a SQL query that joins the `SalesTransaction` and `Customer` tables on `CustomerID` and returns `CustomerName`, `TDate`, and `StoreID`. Load the result into a DataFrame and print the first 5 rows. If you have not taken an introductory database course, use an AI chatbot to solve this part.

c) Close the database connection.

In [13]:
#a)
df_customers = pd.read_sql_query("SELECT * FROM Customer", connection)
print("Shape:", df_customers.shape)
print()
df_customers.head()


Shape: (10, 3)



,customerid,customername,customerzip
0,1-2-333,Tina,60137
1,2-3-444,Tony,60611
2,3-4-555,Pam,35401
3,4-5-666,Elly,47374
4,5-6-777,Nora,60640


In [14]:
#b)
query = """
    SELECT c.CustomerName, s.TDate, s.StoreID
    FROM SalesTransaction s
    JOIN Customer c ON s.CustomerID = c.CustomerID
"""

df_joined = pd.read_sql_query(query, connection)
df_joined.head()

,customername,tdate,storeid
0,Tina,2020-01-01,S1
1,Tony,2020-01-01,S2
2,Tina,2020-01-02,S3
3,Pam,2020-01-02,S3
4,Tony,2020-01-02,S3


In [15]:
#c
connection.close()

## Part 4: APIs

### Exercise 7: Fetching data from a public API

a) Use the `requests` library to fetch the latest EUR exchange rates from:  
`https://open.er-api.com/v6/latest/EUR`

Check that the status code is 200 and print the exchange rates for USD, GBP, and SEK.

b) Convert the full rates dictionary into a pandas DataFrame with columns `currency` and `rate`. Print the first 10 rows.

c) Using the Open-Meteo API, fetch the 7-day forecast for Turku (latitude 60.45, longitude 22.27) with `temperature_2m_max` and `precipitation_sum` as daily variables and `Europe/Helsinki` as the timezone. Convert the result into a DataFrame with columns `date`, `temp_max`, and `precipitation`. Print the DataFrame.

In [16]:
#a
import requests
status_code = requests.get("https://open.er-api.com/v6/latest/EUR")
print(f"Status code: {status_code.status_code}")

if status_code.status_code == 200:
    exchange_rates = status_code.json()['rates']
    print(f"1 EUR = {exchange_rates['USD']} USD")
    print(f"1 EUR = {exchange_rates['GBP']} GBP")
    print(f"1 EUR = {exchange_rates['SEK']} SEK")

Status code: 200
1 EUR = 1.177826 USD
1 EUR = 0.870537 GBP
1 EUR = 10.824078 SEK


In [17]:
#b)
df_exchange_rates = pd.DataFrame(
    list(exchange_rates.items()),
    columns = ['currency', 'rate']
)
print(df_exchange_rates[:10])

  currency         rate
0      EUR     1.000000
1      AED     4.325652
2      AFN    75.793504
3      ALL    95.793425
4      AMD   441.416183
5      ANG     2.108350
6      AOA  1117.091844
7      ARS  1593.603540
8      AUD     1.644629
9      AWG     2.108350


In [18]:
#c)
url = "https://api.open-meteo.com/v1/forecast"
params = {
    'latitude': 60.45, 
    'longitude': 22.27,
    'daily': 'temperature_2m_max,precipitation_sum',
    'timezone': 'Europe/Helsinki'
}

response = requests.get(url, params = params)
forecast = response.json()
daily = forecast['daily']

df_forecast = pd.DataFrame({
    'date': daily['time'],
    'temp_max': daily['temperature_2m_max'],
    'precipitation': daily['precipitation_sum']
})
print(df_forecast)

         date  temp_max  precipitation
0  2026-04-17      17.7            0.1
1  2026-04-18      15.0            0.0
2  2026-04-19      12.7            0.0
3  2026-04-20      13.5            0.0
4  2026-04-21      15.7            0.0
5  2026-04-22      13.2            0.0
6  2026-04-23      12.3            0.0


### Exercise 8: Using an API key with Gemini

This exercise uses the Gemini API key that you stored in `.env` in Exercise 1.

a) Load your API key from `.env` using either approach from Part 1.

b) Install the `google-genai` package (if needed) and use it to send a simple prompt to Gemini. For example, ask it to explain what an API is in one sentence. Print the response.

```python
from google import genai

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain what an API is in one sentence."
)
print(response.text)
```

c) Write a function called `ask_gemini(prompt)` that takes a prompt string as input, sends it to the Gemini API, and returns the response text. Test it with a prompt of your choice.

In [21]:
#a)
api_key = os.environ.get("GEMINI_API_KEY")

import sys
!{sys.executable} -m pip install -q -U google-genai


In [20]:
#b)
from google import genai
help(google)

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain what an API is in one sentence."
)
print(response.text)

Help on package google:

NAME
    google

PACKAGE CONTENTS
    auth (package)
    genai (package)
    oauth2 (package)
    protobuf (package)

FILE
    (built-in)


An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and exchange data with one another.


In [23]:
#c)

def ask_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )
    return response.text

ask_gemini("Hello")

'Hello! How can I help you today?'

## Part 5: Joining and Reshaping Data

### Exercise 9: Merging DataFrames

Consider the following two DataFrames:

```python
stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"],
    "region": ["South", "South", "Central", "North"]
})

sales = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003, 1004, 1005],
    "store_id": [1, 2, 2, 3, 5],
    "amount": [250, 120, 310, 180, 90]
})
```

a) Perform an **inner join** on `store_id`. How many rows are in the result and why?

b) Perform a **left join** with `stores` on the left. Which store has no sales and how is it represented?

c) Perform an **outer join**. Identify which rows have `NaN` values and explain why.

In [29]:
stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"],
    "region": ["South", "South", "Central", "North"]
})

sales = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003, 1004, 1005],
    "store_id": [1, 2, 2, 3, 5],
    "amount": [250, 120, 310, 180, 90]
})

#a)
inner_join = pd.merge(stores, sales, on="store_id", how="inner")
display(inner_join)
print()
print(f"Number of rows: {len(inner_join)}")

,store_id,city,region,transaction_id,amount
0,1,Helsinki,South,1001,250
1,2,Turku,South,1002,120
2,2,Turku,South,1003,310
3,3,Tampere,Central,1004,180



Number of rows: 4


In [33]:
#b)
left = pd.merge(stores, sales, how="left", on="store_id")
display(left)
print()
print("The store with no sales is Oulu (store_id 4) shown with NaN values")

,store_id,city,region,transaction_id,amount
0,1,Helsinki,South,1001.0,250.0
1,2,Turku,South,1002.0,120.0
2,2,Turku,South,1003.0,310.0
3,3,Tampere,Central,1004.0,180.0
4,4,Oulu,North,NaN,NaN



The store with no sales is Oulu (store_id 4) shown with NaN values


In [34]:
#c)
outer_join = pd.merge(stores, sales, how="outer", on="store_id")
display(outer_join)

,store_id,city,region,transaction_id,amount
0,1,Helsinki,South,1001.0,250.0
1,2,Turku,South,1002.0,120.0
2,2,Turku,South,1003.0,310.0
3,3,Tampere,Central,1004.0,180.0
4,4,Oulu,North,NaN,NaN
5,5,NaN,NaN,1005.0,90.0


### Exercise 10: Concatenation and reshaping

a) Create two DataFrames representing monthly sales for two quarters:

```python
q1 = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "store_A": [20000, 22000, 21000],
    "store_B": [15000, 16000, 17000]
})

q2 = pd.DataFrame({
    "month": ["Apr", "May", "Jun"],
    "store_A": [23000, 24000, 25000],
    "store_B": [17500, 18000, 19000]
})
```

Concatenate them vertically into a single DataFrame with `ignore_index=True`. Print the result.

b) Use `melt()` to convert the concatenated DataFrame from wide to long format. The `month` column should remain as the identifier, and the store columns should become rows. Use `store` as the variable name and `sales` as the value name. Print the result.

c) Use `pivot()` to convert the long DataFrame back to wide format. Verify that the result matches the original concatenated DataFrame.

In [37]:
q1 = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "store_A": [20000, 22000, 21000],
    "store_B": [15000, 16000, 17000]
})

q2 = pd.DataFrame({
    "month": ["Apr", "May", "Jun"],
    "store_A": [23000, 24000, 25000],
    "store_B": [17500, 18000, 19000]
})

#a)
frames = [q1, q2]
monthly_sales = pd.concat(frames, ignore_index=True)
print(monthly_sales)

  month  store_A  store_B
0   Jan    20000    15000
1   Feb    22000    16000
2   Mar    21000    17000
3   Apr    23000    17500
4   May    24000    18000
5   Jun    25000    19000


In [42]:
#b)

long = pd.melt(monthly_sales, id_vars = "month", var_name = "store", value_name = "sales")
display(long)

,month,store,sales
0,Jan,store_A,20000
1,Feb,store_A,22000
2,Mar,store_A,21000
3,Apr,store_A,23000
4,May,store_A,24000
5,Jun,store_A,25000
6,Jan,store_B,15000
7,Feb,store_B,16000
8,Mar,store_B,17000
9,Apr,store_B,17500


In [57]:
#c)
wide = long.pivot(index = "month", columns = "store", values = "sales").rename_axis(columns=None).reset_index()

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
wide["month"] = pd.Categorical(wide["month"], categories=month_order, ordered=True)
wide = wide.sort_values("month").reset_index(drop=True)

display(wide)

,month,store_A,store_B
0,Jan,20000,15000
1,Feb,22000,16000
2,Mar,21000,17000
3,Apr,23000,17500
4,May,24000,18000
5,Jun,25000,19000
